## Notebook to generate baselines for fidelity metrics

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
# Root folders
EXPERIMENTS_ROOT = Path("/data/shared/fsibilla/clean_code/Q0/experiments")
BASELINE_OUTPUT_ROOT = Path("/data/shared/fsibilla/clean_code/Q0/fidelity/baselines/nga_micron/marginal")
BASELINE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Seeds
SEEDS = [1, 2, 3, 4, 5]

# Dataset and source model for real data
DATASET_NAME = "nga_micron"
REFERENCE_REAL_MODEL = "TVAE"

# Real file naming
REAL_FILENAME_TEMPLATE = "/data/shared/fsibilla/clean_code/Q1/experiments/nga_micron/full.csv"

# Variables used in the marginal metric
TARGET_COLS = [
    'va_ai', 'fol_ai', 'vb12_ai',
    'fe_ai', 'zn_ai',
    'avg_adult_education', 'log_exp'
]

ADM1_COL = "adm1name"

In [3]:
def load_true_data(seed: int, reference_model: str = REFERENCE_REAL_MODEL) -> pd.DataFrame:
    """
    Load the true/training dataset for the given seed from a reference model folder.
    """
    real_path = (
        EXPERIMENTS_ROOT
        / reference_model
        / DATASET_NAME
        / "results"
        / f"seed_{seed}"
        / REAL_FILENAME_TEMPLATE.format(seed=seed)
    )

    if not real_path.exists():
        raise FileNotFoundError(f"True dataset not found: {real_path}")

    df = pd.read_csv(real_path)

    required_cols = TARGET_COLS + [ADM1_COL]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in true dataset for seed {seed}: {missing}")

    df = df[required_cols].copy()
    df[ADM1_COL] = df[ADM1_COL].astype(str)

    for col in TARGET_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

In [4]:
def make_upper_bound_bootstrap(df_true: pd.DataFrame, seed: int) -> pd.DataFrame:
    """
    Upper bound baseline:
    bootstrap sample of the original dataset, same number of rows, sampled with replacement.
    """
    ub = df_true.sample(
        n=len(df_true),
        replace=True,
        random_state=seed
    ).reset_index(drop=True)

    return ub


def make_lower_bound_mean_dataset(df_true: pd.DataFrame) -> pd.DataFrame:
    """
    Lower bound baseline:
    keep the same number of rows and the same adm1 assignment as the true dataset,
    but replace each numerical variable with its global mean value.
    """
    lb = df_true[[ADM1_COL]].copy().reset_index(drop=True)

    mean_values = df_true[TARGET_COLS].mean(axis=0)

    for col in TARGET_COLS:
        lb[col] = mean_values[col]

    # Reorder columns
    lb = lb[TARGET_COLS + [ADM1_COL]]

    return lb

In [5]:
for seed in SEEDS:
    print(f"Processing seed {seed}...")

    df_true = load_true_data(seed=seed)

    ub_df = make_upper_bound_bootstrap(df_true=df_true, seed=seed)
    lb_df = make_lower_bound_mean_dataset(df_true=df_true)

    seed_out_dir = BASELINE_OUTPUT_ROOT / f"seed_{seed}"
    seed_out_dir.mkdir(parents=True, exist_ok=True)

    ub_path = seed_out_dir / "UB.csv"
    lb_path = seed_out_dir / "LB.csv"

    ub_df.to_csv(ub_path, index=False)
    lb_df.to_csv(lb_path, index=False)

    print(f"  saved UB -> {ub_path}")
    print(f"  saved LB -> {lb_path}")

print("Done.")

Processing seed 1...
  saved UB -> /data/shared/fsibilla/clean_code/Q0/fidelity/baselines/nga_micron/marginal/seed_1/UB.csv
  saved LB -> /data/shared/fsibilla/clean_code/Q0/fidelity/baselines/nga_micron/marginal/seed_1/LB.csv
Processing seed 2...
  saved UB -> /data/shared/fsibilla/clean_code/Q0/fidelity/baselines/nga_micron/marginal/seed_2/UB.csv
  saved LB -> /data/shared/fsibilla/clean_code/Q0/fidelity/baselines/nga_micron/marginal/seed_2/LB.csv
Processing seed 3...
  saved UB -> /data/shared/fsibilla/clean_code/Q0/fidelity/baselines/nga_micron/marginal/seed_3/UB.csv
  saved LB -> /data/shared/fsibilla/clean_code/Q0/fidelity/baselines/nga_micron/marginal/seed_3/LB.csv
Processing seed 4...
  saved UB -> /data/shared/fsibilla/clean_code/Q0/fidelity/baselines/nga_micron/marginal/seed_4/UB.csv
  saved LB -> /data/shared/fsibilla/clean_code/Q0/fidelity/baselines/nga_micron/marginal/seed_4/LB.csv
Processing seed 5...
  saved UB -> /data/shared/fsibilla/clean_code/Q0/fidelity/baselines/ng